In [2]:
#using [GithubRepositoryDataReader](https://github.com/alexeygrigorev/gitsource)
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)


In [3]:
#see document structure

doc = documents[1]
print(f"{doc['filename']}: {doc.get('title', 'No title')}")
print(doc)

01-agentic-rag/lessons/02-environment.md: No title
{'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following:\n\n- Python (3.14 or later)\n- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible\n  provider like Groq, Gemini, or Ollama)\n- Basic familiarity with Python and the command line\n\n## Creating the project\n\nWe\'ll start from scratch - no cloning needed. You\'ll create the\nproject yourself, step by step.\n\nFirst, install uv. It\'s a Python package manager, and I switched all my\nprojects to it because it\'s fast and convenient. Once I started using\nit, I never wanted to go back.\n\nOn Mac or Linux:\n\n```bash\ncurl -LsSf https://astral.sh/uv/install.sh | sh\n```\n\nOn Windows:\n\n```powershell\npowershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"\

In [17]:
#Q1. How many lesson pages

total_documents = len(documents)
print(total_documents)

72


In [6]:
#Q2. Indexing and searching

from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index


In [19]:
#Q2. searching

docs_index = build_index(documents)

query = "How does the agentic loop keep calling the model until it stops?"

search_results = docs_index.search(
    query=query,
    filter_dict=None,
    boost_dict=None,    
    num_results=5
)

for result in search_results:
    print(f"Filename: {result['filename']}")
    #print(f"Content: {result['content'][:50]}")  # Print the first 50 characters of the content
    #print("-" * 80)


Filename: 01-agentic-rag/lessons/14-agentic-loop.md
Filename: 01-agentic-rag/lessons/15-frameworks.md
Filename: 01-agentic-rag/lessons/13-function-calling.md
Filename: 01-agentic-rag/lessons/11-agents-intro.md
Filename: 01-agentic-rag/lessons/16-other-frameworks.md


In [20]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

from agent_client.rag import RAGBase
assistant = RAGBase(docs_index, openai_client)

cntx = assistant.build_context(search_results, result_attributes=['filename', 'content'])

print(cntx)

=== DOCUMENT START ===
FILENAME:
01-agentic-rag/lessons/14-agentic-loop.md
CONTENT:
# The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Tools,

In [21]:
answer = assistant.rag(query)

print(answer.output_text)


The loop keeps calling the model by checking whether the latest response included any `function_call` items.

- It sends the current `messages` history to the model.
- If the model returns a function call, the code runs the tool, appends the tool output, and sets `has_function_calls = True`.
- After processing the response, if `has_function_calls == False`, it `break`s out of the `while True` loop.

So it keeps going until the model returns a response with no function calls.


In [22]:
# Q3. Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

print(answer.usage.input_tokens)

7214


In [4]:
#Q4. Chunking - How many chunks do you get?

from gitsource import chunk_documents

chunks_of_docs = chunk_documents(documents, size=2000, step=1000)

print(len(chunks_of_docs))


295


In [24]:
#Q5. RAG with chunking

chunks_index = build_index(chunks_of_docs)
assistant_with_chunks = RAGBase(chunks_index, openai_client)

answer_with_chunks = assistant_with_chunks.rag(query)

print(answer_with_chunks.output_text)
print(answer_with_chunks.usage.input_tokens)

It keeps a `while True` loop and checks whether the model returned any `function_call` items.

- If there are function calls, the code runs them, appends the tool results to `messages`, and loops again.
- If there are no function calls on that turn, it breaks out of the loop and stops.

So the stop condition is: **no function calls returned in the latest model response**.
2397


In [ ]:
print(answer_with_chunks.usage.input_tokens)

print("{}{}".format('Reduction on input tokens: ', str(answer.usage.input_tokens / answer_with_chunks.usage.input_tokens) + ' times'))

2397
Reduction on input tokens:3.009595327492699 times


In [ ]:
# Q6. Turning it into an agent

## Create a search function that uses the chunk index.

chunks_index = build_index(chunks_of_docs)

def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return chunks_index.search(
        query,
        filter_dict=None,
        boost_dict=None, 
        num_results=5,
    )

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_tools = Tools()
agent_tools.add_tool(search)

agent_tools.get_tools()



[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [9]:
instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()


query = "How does the agentic loop work, and how is it different from plain RAG?"

In [10]:
# The chat interface and runner

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [11]:
# Running one prompt

result = runner.loop(
    prompt=query,
    callback=callback,
)

-> Response received


-> Response received
